# Session 3: break-even with a mix, steps, uncertainty and time
**A. Real-world problem.** The made-up bakery of Session 2 is more complicated than one
product with one price. It sells bread *and* cakes, it must buy a second oven when it grows,
its prices, costs and customers are never exactly what the plan says, and the start-up money
is paid back slowly, over months. A single break-even number hides all of that. Today you
build the richer model, check it against a formula, and ask which input matters most.

*General education only: the bakery and its numbers are made up, and this is not
financial advice.*

In [ ]:
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

F, P, V, Q = 1200.0, 3.50, 1.40, 800.0          # as in Session 2
DEMAND_A, DEMAND_B = 1675.0, 250.0              # loaves per month = 1675 - 250 * price
PRODUCTS = {"loaf": (3.50, 1.40), "cake": (18.0, 9.0)}   # (price, variable cost) in dollars each

## Step 1: several products and a sales mix
The **sales mix** says what share of the *units* sold each product has. The weighted
contribution margin is the mix-weighted average of (p - v). Break-even units for the
whole bakery: F / weighted margin. Changing the mix changes the answer even if F does not.

In [ ]:
def weighted_cm(mix):
    """Weighted contribution margin, dollars per unit sold; mix = {product: share of units}."""
    if abs(sum(mix.values()) - 1.0) > 1e-9:
        raise ValueError("mix shares must add up to 1")
    return sum(share * (PRODUCTS[name][0] - PRODUCTS[name][1]) for name, share in mix.items())


def multi_break_even(mix, fixed=F):
    """Total break-even units and units of each product (None if the mix earns nothing)."""
    w = weighted_cm(mix)
    if w <= 0:
        return None
    total = fixed / w
    return total, {name: share * total for name, share in mix.items()}

## Step 2: step-fixed costs (a second oven)
One oven bakes at most 700 loaves a month and costs $1200 a month in all. Above 700 loaves a
second oven adds $900. So the fixed cost jumps, and the profit line jumps down with it.
That can create *several* crossings of zero. We find them by scanning whole loaves.

In [ ]:
def step_fixed(q, base=F, extra=900.0, cap=700):
    """Fixed cost needed to bake q loaves: base for the first oven, `extra` for each further one."""
    ovens = max(1, int(math.ceil(q / cap - 1e-12)))
    return base + extra * (ovens - 1)


def profit_step(q, p=P, v=V, base=F, extra=900.0, cap=700):
    return (p - v) * q - step_fixed(q, base, extra, cap)


def sign_changes(fn, q_max=1400):
    """Scan q = 0, 1, 2, ... and report each place where 'profit >= 0' flips: (q_before, q_after, 'up' or 'down')."""
    out = []
    prev = fn(0) >= -1e-9
    for q in range(1, q_max + 1):
        now = fn(q) >= -1e-9
        if now != prev:
            out.append((q - 1, q, "up" if now else "down"))
        prev = now
    return out


def step_break_evens(p=P, v=V, base=F, extra=900.0, cap=700, ovens_max=2):
    """Formula for each oven count k: q_k = F_k / (p - v), kept only if it lies inside that step."""
    res = []
    for k in range(1, ovens_max + 1):
        qk = (base + extra * (k - 1)) / (p - v)
        if (k - 1) * cap < qk <= k * cap:
            res.append(qk)
    return res

## Step 3: uncertainty (Monte Carlo)
Price, variable cost and demand are never exactly as planned. We draw 20000 random
"months" from normal distributions (seeded, so every run gives the same numbers) and
compute the profit of each one. The share of months with profit < 0 is the loss probability.

In [ ]:
def simulate_profit(n=20000, seed=42, p_mu=P, p_sd=0.20, v_mu=V, v_sd=0.15, d_mu=Q, d_sd=150.0, fixed=F):
    rng = np.random.default_rng(seed)
    price = rng.normal(p_mu, p_sd, n)
    vcost = rng.normal(v_mu, v_sd, n)
    demand = np.maximum(rng.normal(d_mu, d_sd, n), 0.0)
    return (price - vcost) * demand - fixed


def summarize(profits):
    return {"mean": float(np.mean(profits)), "std": float(np.std(profits)),
            "p5": float(np.percentile(profits, 5)), "p50": float(np.percentile(profits, 50)),
            "p95": float(np.percentile(profits, 95)), "loss": float(np.mean(profits < 0))}


def loss_probability_demand_only(d_mu=Q, d_sd=150.0, fixed=F, p=P, v=V):
    """Reference formula when only demand is random: P(q < q*) = Phi((q* - mean) / sd)."""
    z = (fixed / (p - v) - d_mu) / d_sd
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))

## Step 4: which input matters most (tornado)
Move one input at a time by -10 % and +10 %, keep the others at plan, and record the profit.
The input with the widest swing is the one to watch.

In [ ]:
def model_profit(p=P, v=V, q=Q, fixed=F):
    return (p - v) * q - fixed


def tornado(pct=0.10):
    """List of (name, profit_low, profit_high, swing) sorted by swing, largest first."""
    base = {"price": P, "variable cost": V, "demand": Q, "fixed cost": F}
    keys = {"price": "p", "variable cost": "v", "demand": "q", "fixed cost": "fixed"}
    rows = []
    for name, value in base.items():
        lo = model_profit(**{keys[name]: value * (1 - pct)})
        hi = model_profit(**{keys[name]: value * (1 + pct)})
        rows.append((name, lo, hi, abs(hi - lo)))
    return sorted(rows, key=lambda r: -r[3])

## Step 5: time value - payback and net present value
Start-up money is spent now; profit arrives month by month. A dollar next year is worth a
little less than a dollar today, so we *discount* each month's cash by the monthly rate.
(The openloan repo converts rates the same way for loans.) Payback = first month in which
the running total reaches the start-up cost. NPV = discounted cash received - start-up cost.

In [ ]:
def monthly_rate(annual):
    return (1.0 + annual) ** (1.0 / 12.0) - 1.0


def cumulative_cash(startup, flows, annual_rate=0.0):
    """Running total after each month, starting from -startup."""
    r = monthly_rate(annual_rate)
    t = np.arange(1, len(flows) + 1)
    return -startup + np.cumsum(np.asarray(flows, dtype=float) / (1.0 + r) ** t)


def payback_month(startup, flows, annual_rate=0.0):
    """First whole month with running total >= 0 (None if it never pays back in the horizon)."""
    ok = np.nonzero(cumulative_cash(startup, flows, annual_rate) >= -1e-9)[0]
    return int(ok[0]) + 1 if len(ok) else None


def npv(startup, flows, annual_rate):
    return float(cumulative_cash(startup, flows, annual_rate)[-1])

## Step 6: failure cases
(a) If the price is not above the variable cost there is *no* break-even; the planner must
say so. (b) If demand falls as price rises, there may be two break-even prices, or none.

In [ ]:
def demand(price):
    return np.maximum(DEMAND_A - DEMAND_B * np.asarray(price, dtype=float), 0.0)


def profit_at_price(price, fixed=F, v=V):
    return (np.asarray(price, dtype=float) - v) * demand(price) - fixed


def break_even_prices(fixed=F, v=V):
    """Prices where profit = 0 with linear demand: roots of (p - v)(a - b p) = F. Empty list if none."""
    b1 = (DEMAND_A + DEMAND_B * v) / DEMAND_B
    c1 = (DEMAND_A * v + fixed) / DEMAND_B
    disc = b1 * b1 - 4 * c1
    if disc < 0:
        return []
    r = math.sqrt(disc)
    return [(b1 - r) / 2, (b1 + r) / 2]


def plan(fixed, p, v):
    """What a careful planner says: the break-even, or the reason there is none."""
    if p <= v:
        return f"No break-even: the price {p:.2f} does not cover the variable cost {v:.2f}."
    return f"Break-even at {fixed / (p - v):.1f} units."

## Step 7: choose price and capacity (the mini-project engine)
Three oven sizes: capacity (loaves/month) -> (fixed cost per month, start-up cost).
Sales = min(demand, capacity). For each choice we report break-even, margin of safety,
loss probability and payback.

In [ ]:
OPTIONS = {600: (1000.0, 5000.0), 800: (1300.0, 6500.0), 1000: (1600.0, 8000.0)}


def evaluate(price, capacity, n=20000, seed=7, v_mu=V, v_sd=0.15, d_sd=150.0, horizon=36):
    fixed, startup = OPTIONS[capacity]
    rng = np.random.default_rng(seed)
    vcost = rng.normal(v_mu, v_sd, n)
    want = np.maximum(rng.normal(float(demand(price)), d_sd, n), 0.0)
    sold = np.minimum(want, capacity)
    pr = (price - vcost) * sold - fixed
    mean_profit = float(np.mean(pr))
    q_star = fixed / (price - v_mu) if price > v_mu else None
    mean_sold = float(np.mean(sold))
    mos = None if q_star is None else (mean_sold - q_star) / mean_sold
    return {"price": price, "capacity": capacity, "break_even": q_star, "mean_sold": mean_sold,
            "mos": mos, "mean_profit": mean_profit, "loss": float(np.mean(pr < 0)),
            "payback": payback_month(startup, [mean_profit] * horizon)}


def passes(r):
    """Mini-project rule: loss probability <= 5 %, payback within 12 months, margin of safety >= 20 %."""
    return r["loss"] <= 0.05 and r["payback"] is not None and r["payback"] <= 12 and r["mos"] is not None and r["mos"] >= 0.20

## See it
Three figures: the broken-up profit line with its crossings, the Monte Carlo histogram,
and the tornado chart.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    for name, mix in {"80/20": {"loaf": 0.8, "cake": 0.2}, "90/10": {"loaf": 0.9, "cake": 0.1},
                      "60/40": {"loaf": 0.6, "cake": 0.4}}.items():
        tot, parts = multi_break_even(mix)
        print(f"mix loaf/cake {name}: weighted margin {weighted_cm(mix):.2f}, break-even {tot:.1f} units "
              f"({parts['loaf']:.1f} loaves, {parts['cake']:.1f} cakes)")
    print("sign changes:", sign_changes(profit_step), " formula:", [round(x, 2) for x in step_break_evens()])
    prof = simulate_profit()
    s = summarize(prof)
    print({k: round(x, 4) for k, x in s.items()})
    only = simulate_profit(p_sd=0.0, v_sd=0.0)
    print(f"demand only: simulated loss {np.mean(only < 0):.4f}, formula {loss_probability_demand_only():.4f}")
    for row in tornado():
        print(f"tornado {row[0]:14s} low {row[1]:7.1f} high {row[2]:7.1f} swing {row[3]:6.1f}")
    flows = [480.0] * 36
    print("payback", payback_month(6000, flows), "discounted payback", payback_month(6000, flows, 0.12),
          "NPV 24 months", round(npv(6000, flows[:24], 0.12), 2), "undiscounted", round(npv(6000, flows[:24], 0.0), 2))
    print("break-even prices", np.round(break_even_prices(), 3), " F=1800:", break_even_prices(1800.0))
    print(plan(1200, 1.4, 1.4))
    for cap in OPTIONS:
        for price in (3.5, 4.0, 4.25, 4.5):
            r = evaluate(price, cap)
            print(cap, price, round(r["break_even"], 1), round(r["mean_sold"], 1), round(r["mos"], 3),
                  round(r["mean_profit"], 1), round(r["loss"], 3), r["payback"], passes(r))

    # figure 1: step profit
    q = np.arange(0, 1401)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(q, [profit_step(x) for x in q], color="#1f6f8b", lw=1.5)
    ax.axhline(0, color="black", lw=0.8)
    ax.axvline(700, color="gray", ls=":")
    ax.text(705, -1500, "oven full:\nsecond oven needed", fontsize=8)
    for a, b, d in sign_changes(profit_step):
        ax.plot([b], [0], "ro" if d == "down" else "go")
    ax.set_xlabel("loaves sold per month (loaves)")
    ax.set_ylabel("monthly profit ($)")
    ax.set_title("Step-fixed cost: profit crosses zero three times")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "step_profit.png", dpi=120)

    # figure 2: histogram
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.hist(prof, bins=60, color="#2a9d8f", alpha=0.85)
    ax2.axvline(0, color="#d9534f", lw=2)
    ax2.set_xlabel("monthly profit ($)")
    ax2.set_ylabel("number of simulated months (of 20000)")
    ax2.set_title(f"Profit distribution: {100 * s['loss']:.1f} % of months make a loss")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "profit_hist.png", dpi=120)

    # figure 3: tornado
    rows = tornado()[::-1]
    base = model_profit()
    fig3, ax3 = plt.subplots(figsize=(6, 3.6))
    for i, (name, lo, hi, _) in enumerate(rows):
        ax3.barh(i, hi - base, left=base, color="#2a9d8f")
        ax3.barh(i, lo - base, left=base, color="#d9534f")
    ax3.set_yticks(range(len(rows)))
    ax3.set_yticklabels([r[0] for r in rows])
    ax3.axvline(base, color="black", lw=0.8)
    ax3.set_xlabel("monthly profit ($) when the input moves -10 % (red) or +10 % (green)")
    ax3.set_title("Tornado: price matters most")
    ax3.grid(alpha=0.3, axis="x")
    fig3.tight_layout()
    fig3.savefig(OUT / "tornado.png", dpi=120)

    # figure 4: payback
    fig4, ax4 = plt.subplots(figsize=(6, 4))
    months = np.arange(1, 37)
    ax4.plot(months, cumulative_cash(6000, flows), label="no discounting")
    ax4.plot(months, cumulative_cash(6000, flows, 0.12), label="discounted at 12 % a year")
    ax4.axhline(0, color="black", lw=0.8)
    ax4.set_xlabel("months after start-up (months)")
    ax4.set_ylabel("running total ($)")
    ax4.set_title("Payback: 13 months, or 14 with discounting")
    ax4.grid(alpha=0.3)
    ax4.legend()
    fig4.tight_layout()
    fig4.savefig(OUT / "payback.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Run `simulate_profit(seed=1)` and `simulate_profit(seed=2)` and compare their loss probabilities with the seed 42 value. How far apart are they?
2. **Stretch.** Add a third product to `PRODUCTS` (for example a muffin) and find the break-even for a mix of your choice with `multi_break_even`.
3. **Explain.** Explain why profit at 700 loaves is higher than at 701 loaves even though 701 loaves bring in more income, and what an owner could do about it.

## Assessment
- **A. Conceptual.** Why can the break-even of a two-product business go down even though the fixed cost and all prices stay the same?
- **B. Numerical.** A 6000 dollar oven and fit-out earns 480 dollars of cash each month. Find the simple payback month, the discounted payback month at 12 percent a year, and the NPV over 24 months at that rate.
- **C. Graph and simulation interpretation.** In the Monte Carlo histogram, which part of the picture is the loss probability, and what would happen to it if demand became more uncertain (a larger standard deviation) with the same mean?
- **D. Design and optimisation challenge.** Design: the owner may choose the 600, 800 or 1000 loaf oven and a price from 3.00 to 5.00 dollars in 0.25 steps. A choice passes if the loss probability is at most 5 percent, the payback at most 12 months and the margin of safety at least 20 percent. Use `evaluate` to recommend one choice and explain the trade-off.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    # --- multi-product mix
    mix = {"loaf": 0.8, "cake": 0.2}
    tot, parts = L.multi_break_even(mix)
    checks.append(("hand calculation: 0.8*2.10 + 0.2*9.00 = 3.48 per unit, break-even 1200/3.48 = 344.8 units",
                   abs(L.weighted_cm(mix) - 3.48) < 1e-9 and abs(tot - 344.8276) < 1e-3))
    checks.append(("the 80/20 break-even is about 275.9 loaves and 69.0 cakes",
                   abs(parts["loaf"] - 275.86) < 0.01 and abs(parts["cake"] - 68.97) < 0.01))
    t90 = L.multi_break_even({"loaf": 0.9, "cake": 0.1})[0]
    t60 = L.multi_break_even({"loaf": 0.6, "cake": 0.4})[0]
    checks.append(("mix shift: more loaves (90/10) raises break-even to 430.1 units; more cakes (60/40) lowers it to 246.9",
                   abs(t90 - 430.11) < 0.01 and abs(t60 - 246.91) < 0.01 and t60 < tot < t90))
    checks.append(("a single-product mix reproduces Session 2 (571.43 loaves)",
                   abs(L.multi_break_even({"loaf": 1.0, "cake": 0.0})[0] - 571.4286) < 1e-3))
    checks.append(("limit: shares that do not add to 1 are rejected", _raises(lambda: L.weighted_cm({"loaf": 0.5, "cake": 0.2}))))
    # --- step-fixed costs
    sc = L.sign_changes(L.profit_step)
    checks.append(("worked example: profit crosses zero up at 572, down at 701 and up again at 1000 loaves",
                   sc == [(571, 572, "up"), (700, 701, "down"), (999, 1000, "up")]))
    be = L.step_break_evens()
    checks.append(("formula crossings 571.43 and 1000 lie inside the scanned intervals",
                   len(be) == 2 and 571 < be[0] <= 572 and 999 < be[1] <= 1000))
    checks.append(("step fixed cost: $1200 up to 700 loaves, $2100 from 701 to 1400",
                   L.step_fixed(700) == 1200 and L.step_fixed(701) == 2100 and L.step_fixed(1400) == 2100
                   and L.step_fixed(0) == 1200))
    checks.append(("profit is $270 at 700 loaves, then -$627.90 at 701", abs(float(L.profit_step(700)) - 270.0) < 1e-9
                   and abs(float(L.profit_step(701)) + 627.9) < 1e-9))
    # --- Monte Carlo
    s = L.summarize(L.simulate_profit())
    checks.append(("seeded run: same numbers every time", L.summarize(L.simulate_profit()) == s))
    checks.append(("Monte Carlo: mean profit about $476 (plan says $480), standard deviation about $375",
                   abs(s["mean"] - 476.4) < 1.0 and abs(s["std"] - 374.9) < 1.0))
    checks.append(("Monte Carlo: loss probability about 9.5 percent; 5th and 95th percentile about -$102 and $1120",
                   abs(s["loss"] - 0.095) < 0.003 and abs(s["p5"] + 101.9) < 2.0 and abs(s["p95"] - 1120.4) < 3.0))
    only = L.simulate_profit(p_sd=0.0, v_sd=0.0)
    ref = L.loss_probability_demand_only()
    checks.append(("validation: with only demand random the simulation matches the formula (6.5 percent simulated, 6.4 percent by the formula)",
                   abs(ref - 0.0638) < 5e-4 and abs(float(np.mean(only < 0)) - ref) < 0.005 and abs(float(np.mean(only < 0)) - 0.065) < 0.001))
    checks.append(("limit: with no randomness at all the profit is exactly $480 every month",
                   np.allclose(L.simulate_profit(n=50, p_sd=0, v_sd=0, d_sd=0), 480.0)))
    # --- tornado
    tor = L.tornado()
    names = [r[0] for r in tor]
    checks.append(("worked example: tornado ranks price, demand, fixed cost, variable cost",
                   names == ["price", "demand", "fixed cost", "variable cost"]))
    sw = {r[0]: r[3] for r in tor}
    checks.append(("tornado swings are $560, $336, $240 and $224",
                   abs(sw["price"] - 560) < 1e-6 and abs(sw["demand"] - 336) < 1e-6
                   and abs(sw["fixed cost"] - 240) < 1e-6 and abs(sw["variable cost"] - 224) < 1e-6))
    pr = {r[0]: (r[1], r[2]) for r in tor}
    checks.append(("price -10 percent gives profit $200 and +10 percent gives $760",
                   abs(pr["price"][0] - 200) < 1e-6 and abs(pr["price"][1] - 760) < 1e-6))
    # --- time value
    flows = [480.0] * 36
    checks.append(("worked example: simple payback of $6000 at $480 a month is month 13 (12.5 months exactly)",
                   L.payback_month(6000, flows) == 13 and abs(6000 / 480 - 12.5) < 1e-12))
    checks.append(("discounted payback at 12 percent a year is month 14", L.payback_month(6000, flows, 0.12) == 14))
    checks.append(("monthly rate for 12 percent a year is 0.9489 percent", abs(L.monthly_rate(0.12) - 0.0094888) < 1e-6))
    checks.append(("NPV over 24 months: $4259 at 12 percent a year, $5520 with no discounting",
                   abs(L.npv(6000, flows[:24], 0.12) - 4259.15) < 0.01 and abs(L.npv(6000, flows[:24], 0.0) - 5520.0) < 1e-9))
    checks.append(("hand calculation: one payment of $1120 after 12 months is worth $1000 today at 12 percent a year",
                   abs(1120 / (1 + L.monthly_rate(0.12)) ** 12 - 1000.0) < 1e-9))
    checks.append(("limit: $150 a month never repays $6000 within 36 months", L.payback_month(6000, [150.0] * 36) is None))
    # --- failure cases
    checks.append(("failure: price equal to variable cost gives a message, not a number",
                   L.plan(1200, 1.4, 1.4).startswith("No break-even") and L.plan(1200, 1.0, 1.4).startswith("No break-even")
                   and L.plan(1200, 3.5, 1.4) == "Break-even at 571.4 units."))
    bp = L.break_even_prices()
    checks.append(("linear demand: break-even prices are $2.56 and $5.54 and profit is positive between them",
                   len(bp) == 2 and abs(bp[0] - 2.5586) < 1e-3 and abs(bp[1] - 5.5414) < 1e-3
                   and all(abs(float(L.profit_at_price(x))) < 1e-9 for x in bp)
                   and float(L.profit_at_price(4.05)) > 0 > float(L.profit_at_price(2.4)) and float(L.profit_at_price(5.7)) < 0))
    grid = np.arange(1.5, 6.5, 0.01)
    checks.append(("failure: with fixed cost $1800 no price breaks even (best profit is -$44.38)",
                   L.break_even_prices(1800.0) == [] and float(np.max(L.profit_at_price(grid, fixed=1800.0))) < 0
                   and abs(float(L.profit_at_price(4.05, fixed=1800.0)) + 44.375) < 1e-9))
    # --- mini-project
    best = None
    rows = []
    for cap in L.OPTIONS:
        for price in np.arange(3.0, 5.0001, 0.25):
            r = L.evaluate(float(price), cap)
            rows.append(r)
            if L.passes(r) and (best is None or r["mean_profit"] > best["mean_profit"]):
                best = r
    checks.append(("mini-project: the best passing choice is the 600-loaf oven at $4.25",
                   best is not None and best["capacity"] == 600 and abs(best["price"] - 4.25) < 1e-9))
    checks.append(("mini-project: it breaks even at 351 loaves, sells about 546 on average, margin of safety about 36 percent",
                   abs(best["break_even"] - 350.9) < 0.1 and abs(best["mean_sold"] - 545.7) < 1.0 and abs(best["mos"] - 0.357) < 0.005))
    checks.append(("mini-project: loss probability about 4 percent, mean profit about $556, payback 9 months",
                   abs(best["loss"] - 0.042) < 0.004 and abs(best["mean_profit"] - 555.7) < 2.0 and best["payback"] == 9))
    passing = sorted((r["capacity"], r["price"]) for r in rows if L.passes(r))
    checks.append(("mini-project: only $4.00 and $4.25 with the 600-loaf oven pass the rule", passing == [(600, 4.0), (600, 4.25)]))
    r35 = L.evaluate(3.5, 600)
    checks.append(("mini-project: the cheap price $3.50 fails (payback 21 months) and the 1000-loaf oven does not pay back within 36 months at $4.00",
                   r35["payback"] == 21 and not L.passes(r35) and L.evaluate(4.0, 1000)["payback"] is None))
    hits = [(c, round(float(p), 2)) for c in L.OPTIONS for p in np.arange(3.0, 5.0001, 0.05)
            if (lambda r: r["loss"] <= 0.02 and r["payback"] is not None and r["payback"] <= 10)(L.evaluate(float(p), c))]
    checks.append(("roadmap claim: no shipped oven size and price reaches loss <= 2 percent with payback <= 10 months", hits == []))
    return checks


def _raises(fn):
    try:
        fn()
    except ValueError:
        return True
    return False

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
